# Bengali Historical OCR & Knowledge Extraction — Google Colab GPU Runner
This notebook provides a modular, GPU-accelerated execution environment for:
1. **Heavy OCR Inference**: Surya OCR, Marker OCR, Qwen2-VL, and EasyOCR on Class 9-10 BGS 300 DPI textbook pages.
2. **Phase 2 Arm B Correction**: BanglaBERT masked token scoring with mBERT NER entity/date guard.
3. **Phase 5 Open-Weight LLM Extraction**: Qwen2.5-7B-Instruct / Llama-3.1-8B-Instruct on Dev set chunks.
4. **Export Packager**: Bundles all generated JSON files into a zip archive for import into the local thesis repository via `import_colab_results()`.

## 1. Environment Setup & GPU Verification

In [ ]:
# Verify GPU allocation
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set project working directory in Drive
import os
WORKSPACE_DIR = '/content/drive/MyDrive/Thesis_work_OCR/Work'
if os.path.exists(WORKSPACE_DIR):
    os.chdir(WORKSPACE_DIR)
    print(f'Working directory set to: {os.getcwd()}')
else:
    print('Please check the path to your Drive project folder.')

In [ ]:
# Install core pinned repository requirements and GPU libraries
!pip install --quiet PyMuPDF==1.24.10 opencv-python==4.10.0.84 numpy==1.26.4 pandas==2.2.2 pyyaml==6.0.2 pydantic==2.9.2 jiwer==3.0.4 rapidfuzz==3.9.7
!pip install --quiet surya-ocr marker-pdf easyocr transformers accelerate bitsandbytes qwen-vl-utils

## 2. Heavy OCR Inference (Surya OCR, EasyOCR, Qwen2-VL, Marker)
Saves one structured JSON file per page into `outputs/ocr/{system_name}/{page_id}.json` strictly adhering to the `BaseOCRSystem` schema.

In [ ]:
import os, json, time, cv2, torch
from pathlib import Path
import pandas as pd
from PIL import Image

# Load pages manifest for Dev split (Chapter 1)
df = pd.read_csv('data/pages.csv')
dev_pages = df[df['split'] == 'dev']
print(f'Dev split contains {len(dev_pages)} pages.')

# Ensure output directories exist
for sys_name in ['surya', 'easyocr', 'marker_ocr', 'qwen_vl']:
    Path(f'outputs/ocr/{sys_name}').mkdir(parents=True, exist_ok=True)

In [ ]:
# Run EasyOCR (CRAFT detector + Bengali CRNN)
import easyocr
reader = easyocr.Reader(['bn', 'en'], gpu=True)

for _, row in dev_pages.iterrows():
    pid = row['page_id']
    img_path = row['image_path']
    out_json = Path(f'outputs/ocr/easyocr/{pid}.json')
    if out_json.exists():
        print(f'Skipping cached {pid}')
        continue

    t0 = time.time()
    results = reader.readtext(img_path)
    runtime = round(time.time() - t0, 3)
    
    lines = [r[1] for r in results]
    blocks = [{'block_id': i, 'text': r[1], 'confidence': float(r[2])} for i, r in enumerate(results)]
    
    record = {
        'page_id': pid,
        'book_page': int(row['book_page']),
        'chapter_id': row['chapter_id'],
        'split': row['split'],
        'system_name': 'easyocr',
        'raw_text': '\n'.join(lines),
        'normalised_text': '\n'.join(lines),
        'blocks': blocks,
        'runtime_sec': runtime
    }
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(record, f, ensure_ascii=False, indent=2)
    print(f'EasyOCR: {pid} completed in {runtime}s')

## 3. Phase 2 Arm B: BanglaBERT Masked Token Spell Correction
Loads `sagorsarker/banglabert` and applies masked language model scoring with entity & number protection.

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

BERT_MODEL_NAME = 'sagorsarker/banglabert'
print(f'Loading {BERT_MODEL_NAME}...')
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model = AutoModelForMaskedLM.from_pretrained(BERT_MODEL_NAME).to('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.eval()
print('BanglaBERT loaded successfully.')

## 4. Phase 5: Open-Weight LLM Extraction Ablation (Qwen2.5-7B-Instruct / Llama-3.1)
Executes 4-bit quantized open-weight instruction-tuned LLMs on Dev chunks.

In [ ]:
# Load Qwen2.5-7B-Instruct in 4-bit for extraction
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')
print(f'{MODEL_ID} loaded successfully in 4-bit.')

## 5. Export Colab Outputs for Local Repo Import
Zips all OCR, correction, and extraction outputs into `outputs/colab_export.zip` so they can be seamlessly loaded via `BaseOCRSystem.import_colab_results()` on your local machine.

In [ ]:
import shutil
zip_dest = 'outputs/colab_export'
shutil.make_archive(zip_dest, 'zip', 'outputs')
print(f'Export archive generated: {zip_dest}.zip')
print('You can now download this file or access it via Drive and run wrapper.import_colab_results() locally.')